# Visual Recognition Workbench ? Colab
Upload a ZIP of this repository (GitHub Download ZIP works). Setup, model downloads and training run only when you execute their cells in Colab. Enable a GPU runtime for heavy work. No accuracy is promised.

In [ ]:
from google.colab import files
import zipfile, pathlib, os
uploaded = files.upload()
archive = next(iter(uploaded))
root = pathlib.Path("/content/workbench")
root.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for info in z.infolist():
        target = (root / info.filename).resolve()
        if not target.is_relative_to(root.resolve()):
            raise ValueError("Unsafe archive path")
    z.extractall(root)
project = next(root.rglob("recognition.py")).parent
os.chdir(project)
print(project)

## Explicit Colab setup
This cell downloads Python packages. It has not been run locally. Colab already supplies Torch; keep its matching torchvision build.

In [ ]:
%pip install "streamlit>=1.45,<2" "transformers>=4.53,<6" pillow numpy
import torch, torchvision
print(torch.__version__, torchvision.__version__, "GPU:", torch.cuda.is_available())

## Lightweight verification

In [ ]:
!python -m unittest discover -v

## CLIP recognition
This optional cell downloads approximately 600 MB of model weights in Colab. Scores are relative candidate scores, not calibrated accuracy.

In [ ]:
from encoders import CLIPEncoder
from recognition import build_classifier, rank
from PIL import Image
encoder = CLIPEncoder(allow_download=True)
query_files = files.upload()
query_path = next(iter(query_files))
labels = ["cat", "dog", "car", "apple"]
texts = [encoder.encode_texts(["a photo of a " + label]) for label in labels]
with Image.open(query_path) as image:
    result = rank(encoder.encode_images([image.convert("RGB")]), build_classifier(texts), labels)
print(result)

## Optional CNN training
Downloads CIFAR-10 and trains a supervised classifier. Run deliberately; this can take substantial time. Validation selects the checkpoint; the official test set is evaluated afterward.

In [ ]:
# Uncomment only when ready for a heavy Colab run.
# from cnn import train_cifar10
# metrics = train_cifar10("/content/cifar10", "/content/cnn.pt", epochs=15, width=64, download=True)
# print(metrics)
# files.download("/content/cnn.pt")

## Interface and artifacts
The local Streamlit app exposes all interactive workflows. Colab cells above expose the actual model operations without a public tunnel. Download checkpoints and results explicitly; keep personal images out of Git. Automatic VQA/LLM candidate discovery and paper benchmark reproduction are not implemented.